### Введение

ООП в Python-е, разумеется, есть.
Более того, все концепции в нём ровно такие же, как были в прошлом году для C++.
Поэтому заново обсуждать базовые принципы не будем.
Вместо этого сверхсжато посмотрим, "а как это будет в Python-е".

In [ ]:
# Это класс. Пока что пустой. Но уже технически корректный.
class MyClass:
    pass


# Это тоже класс. Уже не совсем пустой.
class Alpha:
    # Это *не* конструктор.
    # Его обычно воспринимают как конструктор.
    # И это даже обычно разумно.
    # Но всё-таки это *не* конструктор.
    # Это инициализатор экземпляра, уже созданного ранее "настоящим" конструктором.
    # Лезть внутрь "настоящего" конструктора обычно не надо, поэтому есть __init__
    # Но если однажды потребуется залезть в "настоящий" конструктор, то он называется __new__
    def __init__(self):
        print("Alpha: __init__ called")

    # Это деструктор.
    # Тут всё честно, это "настоящий" деструктор.
    # Из этого следует интересный момент - __del__ отработает, даже если __init__ упадёт.
    # И вот этот момент стоит на всякий случай иметь в виду.
    def __del__(self):
        print("Alpha: __del__ called")

    # А это просто некий метод
    def do_smth(self):
        print("Alpha: method called")


a = Alpha()
a.do_smth()

Alpha: __init__ called
Alpha: method called


Как вы, наверное, помните из курса C++ в прошлом году,
при использовании правильных конструкций языка деструктор для прикладных классов почти всегда получается пустой.
Здесь в целом то же самое. Вам часто нужен __\_\_init\_\___, это штатно и ожидаемо.
А если вдруг обнаружите себя в ситуации, когда правда обоснованно нужен __\_\_del\_\___,
почитайте сразу ещё про __\_\_enter\_\___ и __\_\_exit\_\___ - это логически родственные конструкции,
которые придуманы для классов, внутри которых происходит захват и освобождение ресурсов.   

### Поля

In [ ]:
# У этого класса будут поля
class TestClass:

    # Это поле класса. Примерно как static-поле в C++, хотя и не совсем.
    foo = 42

    # Это конструктор с параметрами
    def __init__(self, a, b):
        # Возникают ещё два поля класса, теперь уже личные для данного экземпляра.
        # Правило хорошего тона - все поля должны возникнуть внутри __init__-а.
        # Хотя технически ничто не мешает создать новые поля внутри других методов.
        self.bar = a
        self.baz = b

        # Так тоже можно писать. Это снова таплы, да.
        #self.bar, self.baz = a, b

In [ ]:
# Создадим пару экземпляров класса
a = TestClass(1, 2)
b = TestClass(3, 4)

print("=== Initial values ===")
# Распечатаем, посмотрим и на поле класса, и на поля экземпляров
for c in [a, b]:
    print(c.foo, c.bar, c.baz)


Alpha: __del__ called
=== Initial values ===
42 1 2
42 3 4


In [ ]:
# Поменяем поле класса
TestClass.foo = 24
# И поля одного из экземпляров тоже
a.bar = -1
a.baz = -2

print("=== Updated values ===")
# Снова на них посмотрим
for c in [a, b]:
    print(c.foo, c.bar, c.baz)

=== Updated values ===
24 -1 -2
24 3 4


In [ ]:
# Попробуем ещё раз поменять "квазистатическое" поле и ещё раз посмотреть на все значения
a.foo = 88
print("=== Surprise ===")
for c in [a, b]:
    print(c.foo, c.bar, c.baz)

=== Surprise ===
88 -1 -2
24 3 4



Интуитивно очевидно, что в полях класса может быть что угодно - в том числе массивы, другие классы и т.д.
Так вот, интуиция в этот раз не подвела. И правда может.

### Наследование

Наследование в Python-е, очевидно, есть

In [ ]:
# Это базовый класс
class A:
    def __init__(self, v=42):
        self.a = v

    def do_smth(self):
      print("I am A")

    def do_smth_else(self):
      print("I am still A")

# А это унаследованный от него
class B(A):
    # Допустим, наследник хочет свой __init__
    def __init__(self):
        # Тогда на его совести вызвать __init__ родителя,
        # иначе логика инита базового класса не выполнится
        super().__init__(1)
        # Дальше можно свой дополнительный инит писать
        self.b = -1

    def do_smth_else(self):
      print("I am B")


Создадим базовый класс, посмотрим на поля и методы

In [ ]:
a = A()
print(a.a)
a.do_smth()
a.do_smth_else()

42
I am A
I am still A


Аналогично посмотрим на унаследованный

In [ ]:
b = B()
print(b.a)
b.do_smth()
b.do_smth_else()

1
I am A
I am B


Ещё сразу посмотрим на логику того, кто кем является при выполнении кода

In [ ]:
print(isinstance(a, A))
print(isinstance(a, B))
print(isinstance(b, A))
print(isinstance(b, B))

True
False
True
True


Множественное наследование тоже есть. Но оставим его за рамками.
Оно правда редко нужно. Сама концепция та же, что в прошлом году. Потребуется - детали реализации прочитаете.

### Public, protected и private

Аналоги public, protected и private есть. Но с нюансами.
Синтаксически они основаны на именовании:
- что начинается с __ - то private,
- что начинается с _ - то protected,
- что начинается без подчёркиваний - то public.

Но держится всё это на сокрытии имён и порядочности участников процесса.

In [ ]:
# Это базовый класс
class A:
    def __init__(self, v):
        # Это его публичное поле
        self.a = v
        # Это protected
        self._b = v
        # А это приватное
        self.__c = v

In [ ]:
# Это унаследованный класс
class B(A):

    # Это его публичный метод
    def do_some_work(self):
        print(self.a)       # Так можно
        print(self._b)      # Так тоже
        #print(self.__c)    # А так нельзя

    # А это приватный
    def __secret(self):
        print("Secret!")


Это внешний код

In [ ]:
b = B(42)

Так нельзя, поле же private

In [ ]:
print(a.__c)

AttributeError: 'A' object has no attribute '__c'

А вот так внезапно можно. Потому что всего лишь сокрытие имён. Главное - знать, где искать.

In [ ]:
print(b._A__c)

42


Так можно

In [ ]:
b.do_some_work()

42
42


Так нельзя

In [ ]:
b.__secret()

AttributeError: 'B' object has no attribute '__secret'

А так опять можно. Потому что опять главное - знать, где искать.

In [ ]:
b._B__secret()

Secret!


### Немного изнанки и служебных методов

In [ ]:
# Немного изнанки и служебных методов

class A:
    def __init__(self, v=42, t="asd"):
        self.data = v
        self.tag = t

    # Это позволяет задать, как будет виден объект глазами, например, в отладчике и консоли
    def __repr__(self):
        return "class A: %d" % self.data

    # А это - во что превратится объект при явном кастовании в строку
    # Если не задать __str__, для этой цели тоже будет использоваться __repr__
    def __str__(self):
        return "Instance of class A with %d inside" % self.data

    # Как проверять экземпляры на равенство
    # Это примерно перегрузка оператора ==
    def __eq__(self, other):
        return self.data == other.data and self.tag == other.tag

    # Ещё есть перегрузка
    # __ne__(self, other)
    # __lt__(self, other)
    # __le__(self, other)
    # __gt__(self, other)
    # __ge__(self, other)

    # И математику при большом желании тоже можно перегружать
    # __add__(self, other)
    # __mul__(self, other)
    # __sub__(self, other)
    # __mod__(self, other)
    # __truediv__(self, other)

    # Есть ещё более нишевые служебные методы в духе индексации, получения размера и т.д.

    # Немного особняком стоит __hash__
    # Он вычисляет хэш для экземпляра класса.
    # А этот хэш используется, когда класс должен быть сложен в set или оказаться ключом в dict-е.
    # (Под капотом set и dict реализованы как хэш-таблицы. Так что нужен хэш для объекта, чтобы его туда сложить.)
    def __hash__(self):
        # Здесь сейчас сказано, что хэш считается по таплу, в который включены два поля.
        # То есть хэши для двух экземпляров класса будут разные, если значение хотя бы одно из полей у них разное.
        # Хэши совпадут, если значения обоих полей совпадёт.
        # Логически это как будто очень близко к __eq__, но технически используется для совсем других целей.
        return hash((self.data, self.tag))


Попробуйте запускать, меняя __\_\_repr\_\___ и __\_\_str\_\___

In [ ]:

a = A()
print(a)


Instance of class A with 42 inside


In [ ]:
a

class A: 42

А это попробуйте запускать, меняя __\_\_eq\_\___ и значения полей классов

In [ ]:
# А это попробуйте запускать, меняя __eq__ и значения полей классов
b = A(41, "zxc")
c = A(42, "qwe")
d = A()

print(a == b)
print(a == c)
print(a == d)

False
False
True


Поменяем __\_\_eq\_\___:

In [ ]:
class B(A):
  def __eq__(self, other):
        return self.data == other.data

In [ ]:
a = B()
b = B(41, "zxc")
c = B(42, "qwe")
d = B()

print(a == b)
print(a == c)
print(a == d)

False
True
True


#### Challenge

In [ ]:
# Есть у нас вот такой класс
class MyClass:
    def __init__(self, a=0, b=0, c=0):
        self.a, self.b, self.c = a, b, c

    def __repr__(self):
        return "MyClass instance with values (%d; %d; %d)" % (self.a, self.b, self.c)

    # Хэш считается только по полю a
    def __hash__(self):
        return hash(self.a)


То есть хэши вот этих объектов совпадут

In [ ]:
z = MyClass(1, 2, 3)
q = MyClass(1, 8, 42)
print(hash(z))
print(hash(z) == hash(q))


1
True


Попробуем сложить эти объекты в set (хэши совпадают)

In [ ]:
s = set()
s.add(z)
s.add(q)

Запустите, посмотрите на вывод, объясните результат

In [ ]:
# Обойдём set и напечатаем его содержимое
for v in s:
    print(v)


MyClass instance with values (1; 2; 3)
MyClass instance with values (1; 8; 42)


Посмотрим случай с совпадением объектов:

In [ ]:
class OtherClass:
    def __init__(self, a=0, b=0, c=0):
        self.a, self.b, self.c = a, b, c

    def __repr__(self):
        return "OtherClass instance with values (%d; %d; %d)" % (self.a, self.b, self.c)

    # Хэш считается только по полю a
    def __eq__(self, other):
        return self.a == other.a

    def __hash__(self):
        return hash(self.a)



In [ ]:
z = OtherClass(1, 2, 3)
q = OtherClass(1, 8, 42)

s = set()
s.add(z)
s.add(q)


In [ ]:
# Обойдём set и напечатаем его содержимое
for v in s:
    print(v)


OtherClass instance with values (1; 2; 3)


###